# 08 · Chạy ShieldVI V2 Review Dashboard + FastAPI trên Google Colab

Bạn đã test thành công backend với ViSoBERT V1. Notebook này chạy **frontend React thật** và backend chung một địa chỉ `/dashboard/`. **Chỉ dùng dữ liệu demo đã ẩn danh**, không mở API ra Internet công khai.

Lưu ý: Colab sẽ cần kết nối Internet để `npm install` React/Vite. Nếu các thư viện Colab có sẵn xung đột `huggingface-hub`, chỉ cần kiểm tra rằng `transformers==4.57.3` đang được nạp.

**V2 bổ sung cột gán nhãn CLEAN/TOXIC/CHƯA CHẮC và nút Xuất nhãn CSV.**

T015 layout: AI source in `services/moderation-ai/app`, research React in `apps/moderation-web/legacy-v3`. Earlier extraction folders are retained; set REVIEW_DB_PATH explicitly to reuse review history. Social moderation decisions use the separate Social API, not the research annotation DB.


## 1. Cài các thư viện tương thích (chạy một lần, rồi Runtime → Restart session)

Nếu bạn **đã** có `transformers==4.57.3`, có thể bỏ qua ô này. Sau khi restart, **không chạy lại ô cài đặt**.


In [ ]:
%pip install -q "transformers==4.57.3" "huggingface-hub==0.36.2" "tokenizers>=0.22,<0.23" "fastapi>=0.115,<1" "uvicorn[standard]>=0.30,<1" "pydantic>=2.8,<3"


## 2. Kiểm tra phiên bản, import và GPU


In [ ]:
import os
import sys
import json
import zipfile
import tempfile
import subprocess
import time
from pathlib import Path

import torch
import transformers
from google.colab import drive, files, output

print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
assert transformers.__version__ == "4.57.3", "Hãy restart sau khi cài Transformers 4.57.3"
print("Sẵn sàng")


## 3. Upload gói Fullstack ZIP rồi giải nén

Chọn file **`ShieldVI_V2_Review_Fullstack.zip`**, không phải Dataset V3/V4. Nên upload sau lần restart cuối cùng.


In [ ]:
ROOT = Path(tempfile.mkdtemp(prefix='shieldvi_fullstack_', dir='/content'))
ZIP_FILE = Path('/content/ShieldVI_V3_Hybrid_Review.zip')
if not ZIP_FILE.exists():
    print('Chọn ShieldVI_V3_Hybrid_Review.zip')
    selected = files.upload()
    ZIP_FILE = Path('/content') / next(iter(selected))
assert ZIP_FILE.is_file(), ZIP_FILE
# Fresh extraction directory; never remove an earlier session or DB.
with zipfile.ZipFile(ZIP_FILE) as archive:
    for member in archive.infolist():
        target = (ROOT/member.filename).resolve()
        if target != ROOT.resolve() and ROOT.resolve() not in target.parents:
            raise ValueError('ZIP path escapes extraction directory')
    archive.extractall(ROOT)
if not (ROOT/'services/moderation-ai/app/main.py').is_file():
    candidates = [p for p in ROOT.iterdir() if p.is_dir() and (p/'services/moderation-ai/app/main.py').is_file()]
    if len(candidates) == 1: ROOT = candidates[0]
AI_ROOT = ROOT/'services/moderation-ai'
LEGACY_WEB = ROOT/'apps/moderation-web/legacy-v3'
assert (ROOT/'apps/moderation-web/legacy-v3/package.json').exists(), 'Gói ZIP không đúng: thiếu apps/moderation-web/legacy-v3/package.json'
assert (ROOT/'services/moderation-ai/app/main.py').exists(), 'Gói ZIP không đúng: thiếu services/moderation-ai/app/main.py'
print('Đã giải nén project:', ROOT)


## 4. Kết nối Drive chứa mô hình ViSoBERT V1

Nếu bạn đã đổi tài khoản Google, hãy chọn đúng Drive có thư mục `visobert_toxic_final` hoặc sao chép checkpoint sang Drive hiện tại.


In [ ]:
drive.mount('/content/drive')
MODEL_PATH = '/content/drive/MyDrive/visobert_toxic_final'
assert (Path(MODEL_PATH)/'config.json').is_file(), 'Không tìm thấy config.json: kiểm tra tài khoản Drive hoặc MODEL_PATH'
assert (Path(MODEL_PATH)/'threshold_config.json').is_file(), 'Thiếu threshold_config.json'
print('Đã xác nhận checkpoint:', MODEL_PATH)
os.environ['MODEL_PATH'] = MODEL_PATH


## 5. Build React

`npm install` cần mạng để tải React, Vite và thư viện icon. Không dùng `pip install transformers` lại ở bước này.


In [ ]:
print('Node:', subprocess.run(['node','--version'],capture_output=True,text=True).stdout.strip())
print('NPM:', subprocess.run(['npm','--version'],capture_output=True,text=True).stdout.strip())
frontend = LEGACY_WEB
subprocess.run(['npm','install','--no-audit','--no-fund'],cwd=frontend,check=True)
subprocess.run(['npm','run','build'],cwd=frontend,check=True)
assert (ROOT/'services/moderation-ai/app/static/index.html').is_file(), 'Chưa tạo được frontend build'
print('React build thành công: /dashboard/')


## 6. Chạy FastAPI và kiểm tra backend real model

Colab chạy server bên trong runtime hiện tại; không phải dịch vụ production.


In [ ]:
import requests
LOG = ROOT/'uvicorn_colab.log'
env = os.environ.copy()
env['MODEL_PATH'] = MODEL_PATH
env.setdefault('DB_PATH', str(ROOT/'moderation_summaries.sqlite3'))
env.setdefault('REVIEW_DB_PATH', str(ROOT/'shieldvi_reviews.sqlite3'))
server_log = open(LOG,'w',encoding='utf-8')
server = subprocess.Popen(
    [sys.executable,'-m','uvicorn','app.main:app','--host','127.0.0.1','--port','8000','--no-proxy-headers'],
    cwd=AI_ROOT,env=env,stdout=server_log,stderr=subprocess.STDOUT
)
for _ in range(40):
    try:
        r=requests.get('http://127.0.0.1:8000/health',timeout=3)
        if r.status_code==200: break
    except requests.RequestException: pass
    if server.poll() is not None:
        raise RuntimeError('Backend đã dừng. Xem log: '+LOG.read_text(encoding='utf-8')[-2500:])
    time.sleep(2)
else:
    raise RuntimeError('Backend chưa sẵn sàng. Log: '+LOG.read_text(encoding='utf-8')[-2500:])
health=r.json()
print('Health HTTP:',r.status_code)
print(json.dumps(health,ensure_ascii=False,indent=2))
assert health.get('model_loaded'), 'ViSoBERT chưa được nạp: '+str(health.get('details'))
page=requests.get('http://127.0.0.1:8000/dashboard/',timeout=10)
assert page.status_code==200, page.status_code
print('Dashboard HTML:',page.status_code)


## 7. Kiểm tra POST /api/analyze rồi mở Dashboard

Không dùng nội dung thật có tên người dùng/địa chỉ. Bộ demo chỉ gồm mã U01–U03.


In [ ]:
payload=json.loads((ROOT/'examples/sample_thread.json').read_text(encoding='utf-8'))
response=requests.post('http://127.0.0.1:8000/api/analyze',json=payload,timeout=120)
print('Analyze status:',response.status_code)
print(json.dumps({k:v for k,v in response.json().items() if k!='comments'},ensure_ascii=False,indent=2))
assert response.status_code==200, response.text
print('Địa chỉ dashboard bên trong Colab: /dashboard/')
try:
    output.serve_kernel_port_as_iframe(8000, path='/dashboard/', height=850)
except TypeError:
    print('Nếu iframe không hỗ trợ path, lấy link proxy bên dưới:')
    print(output.eval_js('google.colab.kernel.proxyPort(8000)') + '/dashboard/')


### Lưu ý khi demo

- Đừng đóng session Colab nếu đang sử dụng dashboard; khi session kết thúc, backend cũng dừng.
- Không public tunnel FastAPI vì hiện chưa có đăng nhập/rate limit.
- Website chỉ hiển thị cảnh báo nghi vấn, không kết luận chắc chắn một người bắt nạt. Điểm rủi ro là heuristic chưa hiệu chuẩn.
- Để chạy dài hạn, đọc `README.md` và dựng trên máy Windows/local.

- **Xác nhận nhãn**: sau khi phân tích, ở bảng kết quả chọn CLEAN/TOXIC/CHƯA CHẮC. Nhấn **Xuất nhãn CSV** để giữ kết quả trước khi làm mới trang. Nhãn được lưu tạm trong giao diện, không tự lưu trên máy chủ.
